# 05 — Evaluating the RAG Pipeline

Every previous notebook *built* the pipeline. This one asks the question that
actually matters in production: **is it any good, and did my last change help or hurt?**

Until now the only way we judged the pipeline was reading an answer and thinking
"looks about right". That does not scale, does not catch regressions, and cannot
compare two configurations.

## A RAG system has two halves that fail differently

```
question ──▶ RETRIEVAL ──▶ context ──▶ GENERATION ──▶ answer
              │                          │
              └─ did the right           └─ did the model actually
                 passage come back?         use what it was given?
```

If retrieval fails, no model can rescue the answer. If retrieval succeeds but the
model ignores the context, that's a prompt or model problem. **A single quality
score cannot tell these apart, so we measure them separately.**

| | Retrieval metrics | Generation metrics |
|---|---|---|
| How | compare retrieved sources against a golden set | an LLM grades the answer |
| Cost | free, instant | one LLM call per metric |
| Deterministic | yes | no |
| Needs labels | yes | no |
| Run them | on every change | before a release, or on samples |

Ported from the crash-course repo's `1-rag_evaluation.ipynb`, which uses LangSmith
+ OpenAI, to **Groq + local embeddings** so it runs on the free key in `.env`.

## 0. Setup

In [1]:
import sys
sys.path.append("..")

from src.data_loader import load_all_documents
from src.vector_store import FaissVectorStore
from src.evaluation import (
    RetrievalEvaluator, LLMJudge, load_golden_set, evaluate_pipeline, print_report,
)

store = FaissVectorStore(persist_directory="../faiss_store_nb05")
if store.exists():
    store.load()
else:
    store.build_from_documents(load_all_documents("../data"))

Loading embedding model for vector store: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Found 3 PDF files
  loaded attention_is_all_you_need.pdf: 3 pages
  loaded embedding_models_report.pdf: 3 pages
  loaded rag_systems_overview.pdf: 2 pages
Found 2 text files
  loaded machine_learning.txt
  loaded python_intro.txt
Found 1 CSV files
  loaded ai_models.csv: 4 rows
Found 1 JSON files
  loaded rag_faq.json: 4 entries
Found 1 Excel files
  loaded vector_databases.xlsx: 4 rows
Found 1 SQLite databases


  loaded llm_catalog.db: 4 rows from 1 table(s)
Found 1 Word files
  loaded retrieval_evaluation.docx
Total documents loaded: 27
Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384
Split 27 documents into 27 chunks


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape (27, 384)
Vector store now holds 27 vectors
Saved FAISS index + metadata to '../faiss_store_nb05'


## 1. The golden set

Retrieval metrics need ground truth: questions paired with the files that genuinely
contain the answer. Ours has 12 questions covering **all seven file formats** in the
pipeline.

`expected_sources` is a *list* because some facts legitimately live in more than one
file — a hit against any of them counts.

In [2]:
golden = load_golden_set()
print(f"{len(golden)} questions\n")
for item in golden[:4]:
    print(f"[{item['id']}] {item['question']}")
    print(f"      expected: {item['expected_sources']}\n")

12 questions

[q01] What is the attention mechanism?
      expected: ['attention_is_all_you_need.pdf']

[q02] What does multi-head attention allow the model to do?
      expected: ['attention_is_all_you_need.pdf']

[q03] Why are positional encodings needed in a Transformer?
      expected: ['attention_is_all_you_need.pdf']

[q04] What is hard negative mining?
      expected: ['embedding_models_report.pdf']



### One trap worth pointing out

The golden set deliberately lives in `eval/`, **not** in `data/`.

`data/` is the searchable knowledge base, and `load_all_documents()` globs it
recursively — so a golden set stored there gets **ingested into the very index you
are evaluating**. The questions and reference answers become retrievable documents.

This isn't hypothetical: it happened while building this notebook. `golden_qa.json`
was ranking as the *top hit* for several questions, because it literally contained
them. Moving it out of `data/` changed the measured MRR from 0.875 to 0.958 — the
contaminating file had been stealing the rank-1 slot.

**Test-set contamination silently distorts every number you produce.** Keep
evaluation data outside the corpus.

## 2. Retrieval metrics — free and deterministic

Four metrics, each answering a different question:

| Metric | Question it answers |
|---|---|
| **hit rate** | Did the right document appear *anywhere* in the top-k? |
| **MRR** | How *highly* was it ranked? (`1/rank` of the first correct hit, averaged) |
| **recall@k** | What fraction of all relevant documents did we find? |
| **precision@k** | What fraction of what we returned was actually relevant? |

Hit rate and MRR differ in an important way: hit rate treats "ranked 1st" and
"ranked 5th" identically, MRR does not. A system with hit rate 100% and MRR 0.5 is
finding everything but burying it.

In [3]:
evaluator = RetrievalEvaluator(store)
report = evaluator.evaluate(golden, top_k=3)
print_report({"retrieval": report})

RETRIEVAL  (top_k=3, 12 questions)
  hit rate       100.0%   (expected source found at all)
  MRR            0.958   (1.0 = always ranked first)
  recall@k       95.8%
  precision@k    66.7%


### Choosing `top_k` by measuring instead of guessing

Hit rate can only rise as `k` grows — retrieve everything and you always "find" the
answer. Precision falls at the same time. The sweep shows where the trade-off sits.

In [4]:
print(f"{'k':>3}  {'hit rate':>9}  {'MRR':>6}  {'recall':>7}  {'precision':>10}")
print("-" * 44)
for row in evaluator.sweep_top_k(golden, values=[1, 3, 5, 10]):
    print(f"{row['top_k']:>3}  {row['hit_rate']:>8.1%}  {row['mrr']:>6.3f}  "
          f"{row['recall_at_k']:>6.1%}  {row['precision_at_k']:>9.1%}")

  k   hit rate     MRR   recall   precision
--------------------------------------------


  1     91.7%   0.917   83.3%      91.7%
  3    100.0%   0.958   95.8%      66.7%
  5    100.0%   0.958  100.0%      50.0%
 10    100.0%   0.958  100.0%      29.2%


Read that table as a decision, not a curiosity:

- `k=1` already finds most answers, but misses some.
- `k=3` reaches **100% hit rate** — and precision is still reasonable.
- `k=5` and `k=10` add nothing to hit rate or MRR; they only dilute precision.

So `top_k=3` — the pipeline's existing default — is **measurably** the right choice
for this corpus. That's the difference between a tuned parameter and a guessed one.

### Where it still ranks imperfectly

MRR below 1.0 means some question didn't rank its expected source first. The
per-question breakdown says exactly which — this is your debugging entry point.

In [5]:
imperfect = [q for q in report["per_question"] if q["first_hit_rank"] != 1]
print(f"{len(imperfect)} of {len(golden)} questions not ranked first\n")
for q in imperfect:
    print(f"[{q['id']}] first correct hit at rank {q['first_hit_rank']}")
    print(f"      {q['question']}")
    print(f"      expected  {q['expected']}")
    print(f"      retrieved {q['retrieved']}\n")

1 of 12 questions not ranked first

[q11] first correct hit at rank 2
      What is Mean Reciprocal Rank?
      expected  ['retrieval_evaluation.docx']
      retrieved ['rag_faq.json', 'retrieval_evaluation.docx', 'attention_is_all_you_need.pdf']



## 3. Generation metrics — LLM as a judge

Some qualities have no formula. Whether an answer is *grounded* in its context can't be
computed, but a language model can assess it. That's "LLM as a judge" — and there are
**four** standard evaluators, all four defined in the tutor's LangSmith notebook:

| Judge | Compares | Catches | Needs labels? | Scale |
|---|---|---|---|---|
| **Correctness** | answer vs **reference answer** | a factually wrong answer | **yes** | pass/fail |
| **Relevance** | answer vs question | answers that dodge the question | no | 1–5 |
| **Groundedness** | answer vs retrieved context | **hallucination** | no | 1–5 |
| **Retrieval relevance** | context vs question | retriever failures | no | 1–5 |

Two distinctions matter more than the names:

**Correctness is the only one needing labelled data.** It marks the answer against the
`reference_answer` stored in the golden set, like a teacher with a model answer. The other
three need no labels at all — which is why they can run against live production traffic
where nobody wrote down the right answer.

**Correctness and groundedness are not the same question**, and they disagree in both
directions:

- *grounded but incorrect* — faithfully repeats a passage that doesn't actually answer the question
- *correct but ungrounded* — right about the world, but the context never said it, so the pipeline got lucky

Groundedness deliberately punishes the second case. An answer that is true but unsupported
still scores low, because the pipeline didn't earn it.

In [6]:
judge = LLMJudge()   # auto-selects an available Groq model

LLM judge initialized: openai/gpt-oss-120b


### The judges, one at a time

Let's hand each judge an obviously good and an obviously bad case, so you can see
what the scores mean before trusting them on real output.

In [7]:
context = ("The attention mechanism maps a query and a set of key-value pairs to an "
           "output, where the output is a weighted sum of the values.")

print("RELEVANCE — does the answer address the question?")
for answer in ["Attention maps a query and key-value pairs to an output vector.",
               "Python was created by Guido van Rossum in 1991."]:
    v = judge.relevance("What is the attention mechanism?", answer)
    print(f"  score {v['score']}  <- {answer[:52]}...")
    print(f"          {v['reasoning'][:100]}")

RELEVANCE — does the answer address the question?


  score 4  <- Attention maps a query and key-value pairs to an out...
          The answer gives a concise description of attention as mapping queries and key-value pairs to an out


  score 1  <- Python was created by Guido van Rossum in 1991....
          The answer is unrelated to the question about the attention mechanism.


In [8]:
print("GROUNDEDNESS — is every claim supported by the context?")
answers = [
    ("supported",   "The output is a weighted sum of the values."),
    ("unsupported", "Attention was invented in 2017 by Google and requires 8 GPUs to train."),
]
for label, answer in answers:
    v = judge.groundedness(answer, context)
    print(f"  score {v['score']}  ({label}) {answer[:52]}...")
    print(f"          {v['reasoning'][:110]}")

GROUNDEDNESS — is every claim supported by the context?


  score 5  (supported) The output is a weighted sum of the values....
          The answer directly restates a claim explicitly present in the context.


  score 1  (unsupported) Attention was invented in 2017 by Google and require...
          The answer's claims about invention date, creator, and GPU requirements are not mentioned in the provided cont


### Correctness — graded against ground truth

Note the prompt design: the judge is asked for its **explanation first, then the verdict**.
That ordering is deliberate — generating the reasoning before committing to true/false makes
the model work the problem through instead of rationalising a snap decision. The tutor's
notebook makes the same point about field order in its output schema.

Below, four answers to the same question, only two of which are right.

In [9]:
question  = "Which vector database is a fully managed cloud service?"
reference = "Pinecone."

cases = [
    ("exact match",         "Pinecone."),
    ("verbose but right",   "The fully managed cloud option is Pinecone, offered as commercial SaaS."),
    ("wrong answer",        "ChromaDB is the fully managed cloud service."),
    ("dodges the question", "Vector databases store embeddings for similarity search."),
]
for label, answer in cases:
    v = judge.correctness(question, answer, reference)
    print(f"  correct={str(v['correct']):<5}  ({label})")
    print(f"          {v['reasoning'][:110]}")

  correct=True   (exact match)
          The student correctly identified Pinecone, matching the reference answer.


  correct=True   (verbose but right)
          The student correctly identifies Pinecone as the fully managed cloud vector database, matching the reference a


  correct=False  (wrong answer)
          The student claims ChromaDB is the fully managed cloud service, which contradicts the reference answer that id


  correct=False  (dodges the question)
          The student answer does not name Pinecone or any specific fully managed cloud service, so it fails to provide 


## 4. Both halves together

`evaluate_pipeline` scores retrieval deterministically *and* runs all four judges over
generated answers, in one pass.

Three practical notes baked into the function:

- **Correctness is skipped for any golden item with no `reference_answer`.** All twelve of
  ours have one, but the function degrades rather than crashing on a partially labelled set.
- **Rate limits.** Judging one question now costs **five** LLM calls (one answer + four
  judges). Groq's free tier allows 8,000 tokens/minute, which twelve questions will exceed —
  so `call_with_retry` waits out `429`s using the delay the API itself suggests, and `pause`
  spaces requests out.
- **`limit`.** We evaluate 4 questions here to keep this notebook fast. The retrieval
  metrics above already covered all 12 for free.

In [10]:
from src.search import RAGSearch

rag = RAGSearch(store)

full_report = evaluate_pipeline(
    vector_store=store,
    answer_fn=lambda q, k: rag.search_and_summarize(q, top_k=k),
    top_k=3,
    judge=judge,
    limit=4,        # keep the notebook quick; raise for a real run
    pause=3.0,      # free-tier friendly pacing
)
print_report(full_report)

Groq LLM initialized: openai/gpt-oss-120b


RETRIEVAL  (top_k=3, 4 questions)
  hit rate       100.0%   (expected source found at all)
  MRR            1.000   (1.0 = always ranked first)
  recall@k       100.0%
  precision@k    83.3%

GENERATION  (LLM as judge)
  correctness          100.0%   (pass/fail vs reference answer, n=4)
  relevance            5.0    (1-5)
  groundedness         4.75    (1-5)
  retrieval relevance  5.0    (1-5)


In [11]:
for r in full_report["per_question"]:
    print(f"[{r['id']}] {r['question']}")
    for metric in ("relevance", "groundedness", "retrieval_relevance"):
        v = r.get(metric, {})
        print(f"      {metric:20} {v.get('score')}  {v.get('reasoning','')[:88]}")
    print()

[q01] What is the attention mechanism?
      relevance            5  The answer directly explains the attention mechanism by describing its query-key-value f
      groundedness         5  All statements in the answer are directly stated in the context about mapping queries an
      retrieval_relevance  5  The context directly defines the attention mechanism as mapping queries and key‑value pa

[q02] What does multi-head attention allow the model to do?
      relevance            5  The answer directly explains that multi-head attention enables the model to attend to mu
      groundedness         5  The answer directly restates the context sentence that Multi‑Head Attention allows the m
      retrieval_relevance  5  The context explicitly states that multi-head attention allows the model to jointly atte

[q03] Why are positional encodings needed in a Transformer?
      relevance            5  The answer directly explains that positional encodings are needed to provide order infor
      

### What the four evaluators together revealed

On the full 12-question run with all four judges:

| Metric | Result |
|---|---|
| correctness | **100%** (12/12 vs reference answers) |
| relevance | **5.00** |
| groundedness | **4.83** |
| retrieval relevance | **5.00** |

**Correctness at 100% while groundedness sits below 5 is the interesting part.** Every
answer was factually right — yet two of them (`q01`, `q06`) contained a claim the retrieved
context never made. On one run the groundedness judge explained `q01` like this:

> *"The mapping, compatibility, softmax, and weighted sum details are in the context, but
> the claim that this lets the model focus on the most relevant parts is not."*

True about attention; absent from the passage. **Correctness alone would have called this
pipeline flawless.** That is precisely why you run more than one evaluator: correctness
tells you the answer is right, groundedness tells you whether your *retrieval* is why.

### On judge non-determinism

Worth seeing rather than just being warned about: across two runs of the identical
pipeline, `q01`'s groundedness scored **3** and then **4**, moving the average from 4.75 to
4.83. Nothing about the code changed.

This is why the retrieval metrics are the ones to trust numerically, and judge scores are
for spotting regressions and comparing variants. If you need a stable judge number, average
several runs — or use correctness, whose pass/fail verdict is far less jittery than a
five-point scale.

## 5. How to actually use this

**Treat the two metric families differently.** Retrieval metrics are free,
deterministic and trustworthy as numbers — run them on every change. Judge scores
are non-deterministic and biased toward verbose answers; use them to spot
regressions and compare variants, not as ground truth.

**The workflow that pays off:**

1. Record a baseline now, before changing anything.
2. Change exactly one thing — chunk size, `top_k`, embedding model, prompt.
3. Re-run and compare. Keep the change only if the numbers moved the right way.

**Grow the golden set from real failures.** Every time someone reports a bad
answer, add that question to `eval/golden_qa.json` with its correct source. The set
becomes a regression suite that gets more valuable over time. Twelve questions is a
starting point, not a target — aim for 50+ on a real system.

**What to try next, now that you can measure it:**

- Add a cross-encoder reranker (Vectors & Angles, Part 5) and watch MRR move.
- Try the chunking strategies from The Chunking Handbook and compare hit rates.
- Sweep `chunk_size` / `chunk_overlap` the way we swept `top_k`.

Each of those is now a **measurable** experiment rather than a matter of taste.